In [44]:
import requests as rq


prefix = "https://dmg.searcher.studio/api/v1/"
# prefix = "http://0.0.0.0:8080/DMG_2025-06-06/"
# url = prefix + "default/order/filter"
url = prefix + "search/order"
# ?&k=9&lat=51.05&long=3.71&model_ids=KGSearcher0,SemanticSearcher2,VisualSearcher3&filter_text=1997-00&skip=0&limit=20&reverse=false

params = dict(object_ids="1997-0048,6128",
    k=9,lat=51.05, long=3.71, model_ids="KGSearcher0,SemanticSearcher2,VisualSearcher3",
    filter_text="e", skip=0, reverse=True)

res = rq.get(url, params=params).json()
x = res["records"] if (isinstance(res, dict) and ("records" in res)) else res
res_recs = pd.DataFrame.from_records(x).set_index("inventory_number")

In [42]:
res_recs_local = res_recs

In [52]:
res_recs_local.sort_index().order_index

inventory_number
0001_1-2    14565
0001_2-2    12530
0003_0-3    23747
0003_1-3     5893
0003_2-3    17784
            ...  
TUP-0135     2709
TUP-0137     6630
TUP-0139     6921
TUP-0140     9316
TUP-0141    13758
Name: order_index, Length: 24781, dtype: int64

In [53]:
res_recs.sort_index().order_index

inventory_number
0001_1-2    12815
0001_2-2    11243
0003_0-3    22040
0003_1-3    14511
0003_2-3    16966
            ...  
TUP-0135     7868
TUP-0137     3330
TUP-0139    12373
TUP-0140    17681
TUP-0141     4953
Name: order_index, Length: 24824, dtype: int64

In [35]:
url = prefix + "search"

search_res = rq.get(url, params=params)#.json()
# search_res = pd.Series(search_res)

In [37]:
search_res.url

'https://dmg.searcher.studio/api/v1/search?object_ids=1997-0048%2C6128&k=9&lat=51.05&long=3.71&model_ids=KGSearcher0%2CSemanticSearcher2%2CVisualSearcher3&filter_text=e&skip=0&reverse=True'

In [21]:
search_res.sort_values().iloc[::-1]

2017-0432          0.000076
2014-0015          0.000075
2017-0053          0.000075
1990-0048_4-4      0.000075
1996-0025_5-5      0.000075
                     ...   
2013-0033_0-6      0.000014
2009-0011_07-10    0.000013
1976-0165          0.000011
1997-0048          0.000000
6128               0.000000
Length: 24781, dtype: float64

In [22]:
res_recs.sort_values(by="order_index").index 

Index(['2017-0432', '2014-0015', '2017-0053', '1990-0048_4-4', '1996-0025_5-5',
       '1990-0033_7-9', '2153', '4603_2-3', '1987-1190_3-3', '2019-0058_3-3',
       ...
       '1981-0001_0-3', '2009-0011_08-10', '2009-0011_05-10', '2013-0033_1-6',
       '2006-0111', '2013-0033_0-6', '2009-0011_07-10', '1976-0165',
       '1997-0048', '6128'],
      dtype='object', name='inventory_number', length=24781)

In [30]:
res_recs.iloc[search_res.argsort().tolist()]

6128               0.000000
1997-0048          0.000000
1976-0165          0.000011
2009-0011_07-10    0.000013
2013-0033_0-6      0.000014
                     ...   
1996-0025_5-5      0.000075
1990-0048_4-4      0.000075
2017-0053          0.000075
2014-0015          0.000075
2017-0432          0.000076
Length: 24781, dtype: float64

In [31]:
search_res.index

Index(['1992-0004_0-2', '1992-0004_1-2', '1992-0004_2-2', '4521', '5051',
       '5052', '5055', '5057_1-2', '5057_2-2', '5056',
       ...
       '2022-0018_3-3', '2022-0026', '2022-0031_00-11', '2022-0033',
       '2022-0054_2-2', '2022-0028', '2022-0025_0-3', '2022-0025_1-3',
       '2022-0025_2-3', '2022-0025_3-3'],
      dtype='object', length=24781)

In [32]:
res_recs.index

Index(['6128', '1997-0048', '1976-0165', '2009-0011_07-10', '2013-0033_0-6',
       '2006-0111', '2013-0033_1-6', '2009-0011_05-10', '2009-0011_08-10',
       '1981-0001_0-3',
       ...
       '2019-0058_3-3', '1987-1190_3-3', '4603_2-3', '2153', '1990-0033_7-9',
       '1996-0025_5-5', '1990-0048_4-4', '2017-0053', '2014-0015',
       '2017-0432'],
      dtype='object', name='inventory_number', length=24781)

---

In [2]:
from tqdm import tqdm
tqdm.pandas()
from glob import glob

import json
import csv
import numpy as np
import numpy.random as rand
import pandas as pd
from collections import Counter

import rdflib
from rdflib import Graph
from data.data import CollectionAccessor, ImageHandler, EmbeddingSpaceAccessor

from search import Search, Randomiser, Equaliser, GraphSearcher, EmbeddingSearcher, TextEmbeddingSearcher
from moon import MOON, Moon

import requests as rq
import matplotlib.pyplot as plt

In [2]:
from app import init_DMG, init_MKG, order_collection

def lifespan():
    global DMG
    global DMG_searcher
    global DMG_concept_search
    DMG, DMG_searcher, DMG_concept_search = init_DMG()


# def lifespan(): #app: FastAPI):
#     global moon
#     global collections
#     global searches
#     global concept_searches


#     moon = Moon()

#     DMG, DMG_searcher, DMG_concept_search = init_DMG()
#     MKG, MKG_searcher, MKG_concept_search = init_MKG()


#     collections = [DMG, MKG]
#     searches = [DMG_searcher, MKG_searcher]
#     concept_searches = [DMG_concept_search, MKG_concept_search]

#     searches = {c.attrs["id_"]: s for c, s in zip(collections, searches)}
#     concept_searches = {c.attrs["id_"]: cs for c, cs in zip(collections, concept_searches)}
#     collections = {c.attrs["id_"]: c for c in collections}

#     # yield
#     # print("have a lunar day 🌕‬")

lifespan()

[GraphSearcher]: building graph...: 100%|███████████████████████████| 24781/24781 [00:01<00:00, 12890.39it/s]


## dev `search/order/indexof`

In [ ]:
rand_order = df.sample(frac=1.0)

rand_recs = df.sample(4).index
cur_indices = {}
for i in rand_recs:
    bools = (rand_order.index == i)
    if bools.sum() < 1:
        raise ValueError(f"object number {i} is not in the index!")
    if bools.sum() > 1:
        raise ValueError("DUPLICATES!?!?! (this should not happen)")

    cur_indices[i] = int(bools.nonzero()[0][0])


In [ ]:
params = dict(object_ids_index_of="1987-0120_04-14,1987-1343_2-4,3370,3703_0-2",
              object_ids="1987-0120_04-14,1987-1343_2-4,3370,3703_0-2")

order_index("DMG_2025-06-06", **params)

In [ ]:
from app import get_collection

get_collection("DMG_2025-06-06")

---

# order_index stuff

In [41]:
rand_order = DMG.sample(frac=1.0)

In [42]:
order_index = pd.Series(range(len(rand_order)), index=rand_order.index)

In [45]:
rand_obj_nums = DMG.sample(4).index
dict(order_index.loc[rand_obj_nums].items())

{'0224': 23930, '1987-0381_2-2': 8389, '4184': 6923, '0015_2-3': 7448}

---